# Scraping metrics and tracing the three pillars: metrics, logs, traces
> L2 concept exercise — Monitoring & Observability. I wanted to stop treating metrics, logs, and traces as three separate tools and instead see how one request gets recorded in all three. So I built a tiny HTTP service that emits all three signals for every request, then scraped its /metrics endpoint the way a monitoring system would, read its JSON logs, and walked its trace spans — all tied together by a shared trace_id. No external tools, no cloud account: just Python's standard library.

In [ ]:
# last_verified: 2026-08-11 · monitoring-observability-concepts n/a
import threading
import time
import json
import uuid
import random
import urllib.request
import urllib.error
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from collections import defaultdict
from datetime import datetime, timezone

## Building a service that emits all three signals
I wrote a small HTTP server that, for every request to /:
- generates a trace_id (a UUID)
- appends a structured JSON log line tagged with that trace_id
- bumps a request counter and records latency in a histogram
- stores a trace span (start, end, duration) in memory
The server also exposes /metrics in Prometheus text format, so I can scrape it like a real monitoring agent would.

In [ ]:
# last_verified: 2026-08-11 · monitoring-observability-concepts n/a
# In-memory stores: the server writes here, the notebook reads from here
metrics_counters = defaultdict(float)
metrics_histograms = defaultdict(list)  # raw durations, formatted on scrape
log_store = []
trace_store = []

BUCKETS = [0.01, 0.05, 0.1, 0.25, 0.5, 1.0, 2.5, 5.0, 10.0, float('inf')]
SERVICE_NAME = 'mock-api'

class ObservabilityHandler(BaseHTTPRequestHandler):
    def do_GET(self):
        trace_id = str(uuid.uuid4())
        start_mono = time.monotonic()
        start_wall = datetime.now(timezone.utc)

        if self.path == '/metrics':
            body = self._format_metrics()
            self._send(200, body, 'text/plain; version=0.0.4')
            return

        if self.path == '/':
            # Simulate variable work so latency isn't always the same
            time.sleep(random.uniform(0.005, 0.18))
            duration = time.monotonic() - start_mono
            code = 200 if random.random() > 0.15 else 500  # ~15% errors

            # --- metrics ---
            key = f'http_requests_total{{service={SERVICE_NAME},path=/,code={code}}}'
            metrics_counters[key] += 1
            metrics_histograms['http_request_duration_seconds'].append(duration)

            # --- logs (structured, tagged with trace_id) ---
            log_entry = {
                'timestamp': start_wall.isoformat(),
                'level': 'info' if code == 200 else 'error',
                'service': SERVICE_NAME,
                'trace_id': trace_id,
                'path': '/',
                'status': code,
                'message': 'request handled' if code == 200 else 'upstream timeout',
            }
            log_store.append(json.dumps(log_entry))

            # --- traces (span record) ---
            trace_store.append({
                'trace_id': trace_id,
                'service': SERVICE_NAME,
                'span_name': 'handle_request',
                'start': start_wall.isoformat(),
                'duration_seconds': round(duration, 4),
                'status_code': code,
            })

            body = json.dumps({'trace_id': trace_id, 'service': SERVICE_NAME})
            self._send(code, body, 'application/json')
        else:
            log_store.append(json.dumps({
                'timestamp': datetime.now(timezone.utc).isoformat(),
                'level': 'warn', 'service': SERVICE_NAME, 'trace_id': trace_id,
                'path': self.path, 'status': 404, 'message': 'unknown route',
            }))
            self._send(404, json.dumps({'error': 'not found'}), 'application/json')

    def _send(self, code, body, content_type):
        self.send_response(code)
        self.send_header('Content-Type', content_type)
        self.send_header('Content-Length', str(len(body.encode())))
        self.end_headers()
        self.wfile.write(body.encode())

    def _format_metrics(self):
        lines = []
        lines.append('# HELP http_requests_total Total HTTP requests')
        lines.append('# TYPE http_requests_total counter')
        for key, val in sorted(metrics_counters.items()):
            lines.append(f'{key} {val:.0f}')
        lines.append('')
        lines.append('# HELP http_request_duration_seconds Request duration')
        lines.append('# TYPE http_request_duration_seconds histogram')
        durations = metrics_histograms.get('http_request_duration_seconds', [])
        for b in BUCKETS:
            count = sum(1 for d in durations if d <= b)
            b_label = '+Inf' if b == float('inf') else f'{b}'
            lines.append(f'http_request_duration_seconds_bucket{{le="{b_label}"}} {count}')
        lines.append(f'http_request_duration_seconds_sum {sum(durations):.6f}')
        lines.append(f'http_request_duration_seconds_count {len(durations)}')
        return '\n'.join(lines) + '\n'

    def log_message(self, fmt, *args):
        pass  # silence default stderr logging

## Starting the service
I spin up the server on localhost:8080 in a daemon thread so it runs in the background while I send traffic and scrape metrics from the same notebook.

In [ ]:
# last_verified: 2026-08-11 · monitoring-observability-concepts n/a
server = ThreadingHTTPServer(('127.0.0.1', 8080), ObservabilityHandler)
thread = threading.Thread(target=server.serve_forever, daemon=True)
thread.start()
print('mock-api listening on http://127.0.0.1:8080')

## Sending traffic
I fire 10 requests at the service. Each one takes a random amount of time and about 15% return an error, so the metrics and logs will show realistic variation.

In [ ]:
# last_verified: 2026-08-11 · monitoring-observability-concepts n/a
responses = []
for i in range(10):
    try:
        with urllib.request.urlopen('http://127.0.0.1:8080/', timeout=5) as r:
            body = json.loads(r.read())
            responses.append(body)
    except urllib.error.HTTPError as e:
        body = json.loads(e.read())
        body['error_status'] = e.code
        responses.append(body)
    except Exception as e:
        responses.append({'error': str(e)})

print(f'Sent {len(responses)} requests')
for r in responses:
    tid = r.get('trace_id', 'N/A')
    status = 'error' if r.get('error_status') else 'ok'
    print(f'  trace_id={tid}  status={status}')

## Scraping the metrics endpoint
This is the key step that mirrors how Prometheus works: I make an HTTP GET to /metrics and pull the text-format metrics. The same pattern applies to any service that exposes a scrape endpoint.

In [ ]:
# last_verified: 2026-08-11 · monitoring-observability-concepts n/a
with urllib.request.urlopen('http://127.0.0.1:8080/metrics', timeout=5) as r:
    metrics_text = r.read().decode('utf-8')
print(metrics_text)

## Parsing the scraped metrics
I wrote a minimal parser to turn the Prometheus text format back into Python values, then display them in a notebook-friendly way.

In [ ]:
# last_verified: 2026-08-11 · monitoring-observability-concepts n/a
def parse_prometheus(text):
    """Minimal Prometheus text-format parser — enough for counters and histograms."""
    result = {}
    for line in text.strip().split('\n'):
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        if '{' in line:
            name, rest = line.split('{', 1)
            labels, value = rest.rsplit('}', 1)
            value = value.strip()
        else:
            name, value = line.rsplit(' ', 1)
            labels = ''
        result.setdefault(name, {})[labels] = value
    return result

parsed = parse_prometheus(metrics_text)
for metric_name, entries in parsed.items():
    print(f'\n{metric_name}:')
    for labels, val in entries.items():
        label_str = labels if labels else '(no labels)'
        print(f'  {label_str} -> {val}')

## Reading the logs
The service logged every request as a structured JSON line with a trace_id. I can scan those logs for errors and then follow the trace_id into the trace data.

In [ ]:
# last_verified: 2026-08-11 · monitoring-observability-concepts n/a
logs = [json.loads(entry) for entry in log_store]
print(f'Total log entries: {len(logs)}')
errors = [log for log in logs if log.get('level') == 'error']
print(f'Error entries: {len(errors)}')
if errors:
    print('\n--- Error logs ---')
    for log in errors:
        print(f"  [{log['timestamp']}] trace_id={log['trace_id']}  {log['message']}")

## Inspecting a trace
I pick a trace that ended in an error and walk through its span to see how long each step took.

In [ ]:
# last_verified: 2026-08-11 · monitoring-observability-concepts n/a
if errors:
    error_trace_id = errors[0]['trace_id']
    matching_spans = [s for s in trace_store if s['trace_id'] == error_trace_id]
    print(f'Trace {error_trace_id} spans:')
    for span in matching_spans:
        print(f"  {span['span_name']}  duration={span['duration_seconds']}s  status={span['status_code']}")
else:
    print('No errors this run — picking a random trace instead.')
    if trace_store:
        sample = trace_store[0]
        print(f"Trace {sample['trace_id']} spans:")
        print(f"  {sample['span_name']}  duration={sample['duration_seconds']}s  status={sample['status_code']}")

## The trace_id glue

Here is where the three pillars stop being separate tools. The trace_id that the service generated for a request appears in:
- the structured log line (a filter away in any log aggregator)
- the trace span record (the building block of a full trace)
- the latency histogram (the request that trace_id represents is one bucket in the distribution)

In a real system you click a metric spike, filter logs by trace_id, and open the trace — all from one identifier.

In [ ]:
# last_verified: 2026-08-11 · monitoring-observability-concepts n/a
# Pick a trace_id that appears in both logs and traces, then show its metric data
if logs:
    tid = logs[0]['trace_id']
    print(f'trace_id: {tid}')
    print(f"  from log:   path={logs[0]['path']}  status={logs[0]['status']}  msg={logs[0]['message']}")
    matching_span = next((s for s in trace_store if s['trace_id'] == tid), None)
    if matching_span:
        print(f"  from trace: duration={matching_span['duration_seconds']}s  span={matching_span['span_name']}")
    print('  from metric: request counted in http_requests_total and http_request_duration_seconds histogram')

## The three pillars at a glance

| Pillar | What it is | Strength | Weakness |
|---|---|---|---|
| Metrics | Numeric measurements over time (counters, histograms) | Cheap to store, fast to alert on | Loses detail once aggregated |
| Logs | Timestamped text records, preferably structured | Rich detail per event | Hard to query at scale without structure |
| Traces | The journey of one request through all services | Pinpoints latency per hop | Expensive to store every trace |

The shared trace_id is the bridge: metrics tell you *something* is wrong, logs tell you *what* happened, and traces tell you *where* time was spent. Each answers a different question, and together they close the loop.

## What I saw
- Scraping /metrics returns plain text that a parser can split on `{}` and whitespace — nothing magical, but every monitoring system speaks this format.
- The trace_id I generated for each request appeared in both the log entry and the span record. In a real system that identifier is usually injected by an HTTP header (the W3C `traceparent`), but the principle is identical.
- The histogram buckets let me ask "how many requests were under 100ms?" without keeping every individual duration — that is how Prometheus summarizes latency.
- About 15% of requests returned 500, and those error log lines were easy to filter by level. The `code=500` counter would show the same pattern on a dashboard.
- Error rate and latency are the two signals I need to correlate: if the 5xx counter spikes *and* the trace spans are slow, the problem is likely the upstream call, not the network.

## What tripped me up
- **urllib raises HTTPError on 5xx.** My first version treated a 500 response as a normal return and tried to parse the body without catching `urllib.error.HTTPError`. I had to wrap the request loop in try/except to handle both success and error-status bodies.
- **Histogram bucket ordering.** My first attempt sorted buckets alphabetically, which put `+Inf` before `5.0`. Prometheus buckets are cumulative and ordered by the `le` value, so I had to sort them numerically instead.
- **Log volume vs signal.** Ten requests is nothing, but the principle scales: in a real system I would not read logs one-by-one but would pipe them to a log aggregator and filter by trace_id, service, and level.

## What I'd try next
- Swap the in-memory stores for a real metrics backend (push to a local Prometheus via OTLP) and a log aggregator, so the scrape actually hits an external endpoint.
- Add a second service that calls this one, so traces span more than one hop and I can practice multi-span correlation.
- Instrument the notebook itself to emit a span when it scrapes /metrics, so the act of monitoring becomes part of the trace.

In [ ]:
# last_verified: 2026-08-11 · monitoring-observability-concepts n/a
server.shutdown()
print('mock-api stopped')